# Data Cleaning Phase

In [1]:
# 2. Imports
import pandas as pd
import numpy as np
import re
import os
from bs4 import BeautifulSoup
import html
from pathlib import Path

In [2]:
# 3. Configuration / Paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
CLEANED_DIR = DATA_DIR / "cleaned"
REPORTS_DIR = PROJECT_ROOT / "reports"

# Ensure output directories exist
CLEANED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# 4. Load Data
# Load raw datasets safely.
df_medicine = pd.read_csv(RAW_DIR / 'medicine.csv', low_memory=False)
df_generic = pd.read_csv(RAW_DIR / 'generic.csv', low_memory=False)
df_manufacturer = pd.read_csv(RAW_DIR / 'manufacturer.csv', low_memory=False)
df_dosage = pd.read_csv(RAW_DIR / 'dosage form.csv', low_memory=False)
df_drug_class = pd.read_csv(RAW_DIR / 'drug class.csv', low_memory=False)
df_indication = pd.read_csv(RAW_DIR / 'indication.csv', low_memory=False)

In [4]:
# 5. Processing: HTML Cleaning
# The generic descriptions contain HTML tags. This function extracts the full readable text safely.
def clean_html(text):
    if pd.isna(text): return text
    soup = BeautifulSoup(str(text), 'html.parser')
    full_str = soup.find(class_='full-str')
    if full_str:
        clean_text = full_str.get_text(separator=' ')
    else:
        clean_text = soup.get_text(separator=' ')
    clean_text = html.unescape(clean_text)
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()
    return clean_text

html_cols = [col for col in df_generic.columns if 'description' in col and col != 'descriptions count']
for col in html_cols:
    df_generic[f'{col}_clean'] = df_generic[col].apply(clean_html)

## 6. Package Pricing Profile & Parsing Rules
The `package container` field contains varied pricing information. 
Based on profiling, we discovered 5 major patterns:
1. **Unit Price + Pack**: e.g., `Unit Price: ৳ 6.00,(50's pack: ৳ 300.00),`
2. **Description + Pack**: e.g., `10 ml drop: ৳ 7.28,(12's pack: ৳ 87.36),`
3. **Description + Unit**: e.g., `50 ml bottle: ৳ 250.94`
4. **Multiple Units/Packs**: e.g., `10 gm tube: ৳ 70.00,30 gm tube: ৳ 150.00`
5. **No Price / Price Unavailable**

**Rules**:
- We extract `price`, `pack_price`, and `pack_size`.
- For single descriptions (e.g., "5 gm tube"), the price is treated as the unit price for that container.
- For ambiguous/multiple prices, we preserve the string but leave numerical fields NULL to ensure correctness over maximum extraction rate.
- Prices with commas (e.g. ৳ 1,454.37) are cleaned.


In [5]:
# 7. Price Extraction Function
import re

def extract_price_info(pkg_str):
    if pd.isna(pkg_str):
        return pd.Series([None, None, None, None, 'BDT'])
    
    pkg_desc = str(pkg_str).strip()
    price, pack_price, pack_size = None, None, None
    currency = 'BDT'
    
    # 5. No price
    if 'Price Unavailable' in pkg_desc or '৳' not in pkg_desc:
        return pd.Series([pkg_desc, price, pack_price, pack_size, currency])
    
    taka_count = pkg_desc.count('৳')
    
    # 4. Ambiguous cases: multiple unit prices or multiple different sizes without pack info.
    # We choose correctness over guessing, so we leave as NULL.
    if taka_count > 2 or (taka_count == 2 and 'pack:' not in pkg_desc) or (taka_count == 2 and pkg_desc.count('Unit Price') > 1):
        return pd.Series([pkg_desc, price, pack_price, pack_size, currency])
        
    def clean_num(s):
        if not s: return None
        return float(s.replace(',', ''))
        
    # 1 & 2. Unit Price + Pack OR Description + Pack (taka_count == 2)
    if taka_count == 2 and "'s pack:" in pkg_desc:
        m = re.search(r"^(.*?):\s*৳\s*([0-9,.]+).*?\(([\d]+)'s pack:\s*৳\s*([0-9,.]+)\)", pkg_desc)
        if m:
            price = clean_num(m.group(2))
            pack_size = int(m.group(3))
            pack_price = clean_num(m.group(4))
            return pd.Series([pkg_desc, price, pack_price, pack_size, currency])
    
    # 3. Description + Unit ONLY (taka_count == 1)
    if taka_count == 1:
        m = re.search(r'^(.*?):\s*৳\s*([0-9,.]+)', pkg_desc)
        if m:
            price = clean_num(m.group(2))
            return pd.Series([pkg_desc, price, pack_price, pack_size, currency])
            
    return pd.Series([pkg_desc, price, pack_price, pack_size, currency])

# Preserve strength reference
df_medicine['strength_raw'] = df_medicine['strength']
df_medicine['strength_normalized'] = df_medicine['strength_raw'].str.strip()

price_extracted = df_medicine['package container'].apply(extract_price_info)
price_extracted.columns = ['package_description', 'price', 'pack_price', 'pack_size', 'currency']

df_medicine = pd.concat([df_medicine, price_extracted], axis=1)
df_medicine.rename(columns={'package container': 'package_container_raw'}, inplace=True)


In [6]:
# 8. Validation and Extraction Statistics
total_rows = len(df_medicine)
non_null_price = df_medicine['price'].notna().sum()
non_null_pack_price = df_medicine['pack_price'].notna().sum()
non_null_pack_size = df_medicine['pack_size'].notna().sum()

print(f"Total rows: {total_rows}")
print(f"non-null price: {non_null_price}")
print(f"null price: {total_rows - non_null_price}")
print(f"non-null pack_price: {non_null_pack_price}")
print(f"null pack_price: {total_rows - non_null_pack_price}")
print(f"non-null pack_size: {non_null_pack_size}")
print(f"null pack_size: {total_rows - non_null_pack_size}")

print(f"\nprice extraction rate: {non_null_price / total_rows * 100:.2f}%")
print(f"pack_price extraction rate: {non_null_pack_price / total_rows * 100:.2f}%")
print(f"pack_size extraction rate: {non_null_pack_size / total_rows * 100:.2f}%")

print(f"\nQuality Checks:")
print(f"price < 0: {(df_medicine['price'] < 0).sum()}")
print(f"pack_price < 0: {(df_medicine['pack_price'] < 0).sum()}")
print(f"pack_size <= 0: {(df_medicine['pack_size'] <= 0).sum()}")
print(f"pack_price < price: {(df_medicine['pack_price'] < df_medicine['price']).sum()}")


Total rows: 21714
non-null price: 20357
null price: 1357
non-null pack_price: 13527
null pack_price: 8187
non-null pack_size: 13527
null pack_size: 8187

price extraction rate: 93.75%
pack_price extraction rate: 62.30%
pack_size extraction rate: 62.30%

Quality Checks:
price < 0: 0
pack_price < 0: 0
pack_size <= 0: 0
pack_price < price: 0


In [7]:
# 9. Sample Inspection
# Display 30 representative rows containing simple, pack, ambiguous, and unusual cases
cols_to_show = ['brand name', 'package_container_raw', 'package_description', 'price', 'pack_price', 'pack_size', 'currency']
df_medicine[cols_to_show].dropna(subset=['package_container_raw']).sample(30, random_state=42)


,brand name,package_container_raw,package_description,price,pack_price,pack_size,currency
14092,Olmetab,"Unit Price: ৳ 6.00,(50's pack: ৳ 300.00),","Unit Price: ৳ 6.00,(50's pack: ৳ 300.00),",6.00,300.00,50.0,BDT
17830,Sefur DS,50 ml bottle: ৳ 250.94,50 ml bottle: ৳ 250.94,250.94,NaN,NaN,BDT
17984,Servipep,"Unit Price: ৳ 4.00,(50's pack: ৳ 200.00),","Unit Price: ৳ 4.00,(50's pack: ৳ 200.00),",4.00,200.00,50.0,BDT
13099,Naxin,"Unit Price: ৳ 5.02,(50's pack: ৳ 251.00),","Unit Price: ৳ 5.02,(50's pack: ৳ 251.00),",5.02,251.00,50.0,BDT
5552,Desotem,"Unit Price: ৳ 2.40,(100's pack: ৳ 240.00),","Unit Price: ৳ 2.40,(100's pack: ৳ 240.00),",2.40,240.00,100.0,BDT
2692,Bisofar,"Unit Price: ৳ 10.00,(30's pack: ৳ 300.00),","Unit Price: ৳ 10.00,(30's pack: ৳ 300.00),",10.00,300.00,30.0,BDT
5257,Deconil,100 ml bottle: ৳ 80.24,100 ml bottle: ৳ 80.24,80.24,NaN,NaN,BDT
15237,Pegalin ER,"Unit Price: ৳ 45.00,(10's pack: ৳ 450.00),","Unit Price: ৳ 45.00,(10's pack: ৳ 450.00),",45.00,450.00,10.0,BDT
1769,Atrium,"2.5 ml ampoule: ৳ 80.00,(5's pack: ৳ 400.00),","2.5 ml ampoule: ৳ 80.00,(5's pack: ৳ 400.00),",80.00,400.00,5.0,BDT
4274,Civox,60 ml bottle: ৳ 90.35,60 ml bottle: ৳ 90.35,90.35,NaN,NaN,BDT


In [8]:
# 10. Validation: Text Normalization and Deduplication
def strip_ws_and_nans(df):
    for col in df.select_dtypes(['object', 'string']).columns:
        if 'description' not in col and 'package' not in col:
            df[col] = df[col].str.strip()
            
    # Replace common placeholder nulls with actual NaN
    import numpy as np
    df.replace(['N/A', 'NA', 'null', 'None', '', '-'], np.nan, inplace=True)
    return df

datasets = [df_medicine, df_generic, df_manufacturer, df_dosage, df_drug_class, df_indication]
for i in range(len(datasets)):
    datasets[i] = strip_ws_and_nans(datasets[i])
    
    # We remove strictly identical full-row duplicates to ensure row uniqueness
    if datasets[i].duplicated().sum() > 0:
        datasets[i].drop_duplicates(inplace=True)


In [9]:
# 11. Save Output
df_medicine.to_csv(CLEANED_DIR / 'medicine_clean.csv', index=False)
df_generic.to_csv(CLEANED_DIR / 'generic_clean.csv', index=False)
df_manufacturer.to_csv(CLEANED_DIR / 'manufacturer_clean.csv', index=False)
df_dosage.to_csv(CLEANED_DIR / 'dosage_form_clean.csv', index=False)
df_drug_class.to_csv(CLEANED_DIR / 'drug_class_clean.csv', index=False)
df_indication.to_csv(CLEANED_DIR / 'indication_clean.csv', index=False)
print('Data cleaning and saving successfully completed!')


Data cleaning and saving successfully completed!


## 12. Summary
The output is now successfully cleaned and extracted, making it ready for SQL database reloading and Exploratory Data Analysis.
